# Figure 3.3: example of each scenario type

In [ ]:
import sys, random
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from matplotlib.lines import Line2D

GEN_DIR = Path.home() / 'greenhouse_ws' / 'scripts'
sys.path.insert(0, str(GEN_DIR))

import generate_world as g

levels = {"straight": "Straight (0°)", "low": "Low curvature (16°)", "high": "High curvature (32°)"}
dens = {"sparse": "Sparse (2)", "medium": "Medium (5)", "cluttered": "Cluttered (9)"}
SEED = 0

fig, axes = plt.subplots(3, 3, figsize=(15, 11.5), sharex=True, sharey=True)
for ax_row, (c, cname) in zip(axes, levels.items()):
    for ax, (d, dname) in zip(ax_row, dens.items()):
        meta = g.make_world(c, d, SEED)[2]
        rng = random.Random(f"{c}|{d}|{SEED}")          # same random stream as the generator
        plants = g.build_plants(g.Centreline(c, rng), rng)

        for p in plants:
            ax.add_patch(Rectangle((p["x"] - p["size"]/2, p["y"] - p["size"]/2),
                                   p["size"], p["size"], color="#2e7d32"))
        for o in meta["obstacles"]:
            if o["kind"] == "cylinder":
                ax.add_patch(Circle((o["x"], o["y"]), o["radius"], color="#e07b00"))
            else:
                ax.add_patch(Rectangle((o["x"] - o["side"]/2, o["y"] - o["side"]/2),
                                       o["side"], o["side"], color="#e07b00"))

        s, goal = meta["start_pose"], meta["goal_pose"]
        ax.plot(s["x"], s["y"], "o", color="#1f4e9c", ms=11)
        ax.plot(goal["x"], goal["y"], "*", color="#b00020", ms=19)

        ax.set_title(f"{cname} / {dname}" + ("\n(held out from training)" if meta["held_out"] else ""),
                     fontsize=14)
        ax.set_aspect("equal")
        ax.set_xlim(-0.4, 12.4)
        ax.set_ylim(-4.6, 4.6)
        ax.tick_params(labelsize=12)

for ax in axes[2]: ax.set_xlabel("x (m)", fontsize=13)
for ax in axes[:, 0]: ax.set_ylabel("y (m)", fontsize=13)

fig.text(0.5, 0.01, "Green squares: plants     Orange: obstacles in the middle aisle     "
         "Blue dot: start     Red star: goal", ha="center", fontsize=13)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig("figure_3_3_scenario_grid.png", dpi=300, bbox_inches="tight")
plt.show()